# 01. 벡터 RAG: 실제 회수 근거를 먼저 보기

지난주 사용한 ChromaDB 흐름을 작은 문서로 복습합니다. 이 노트북의 검색 벡터는 **한국어 문자 n-gram TF-IDF**입니다. 모델을 내려받지 않아도 같은 절차를 실행할 수 있지만, 최신 의미 임베딩의 품질을 평가하려는 실험은 아닙니다.

관찰 대상은 두 가지입니다. (1) 정책 예외 B가 검색 문맥에서 빠지면 어떤 답의 근거가 부족해지는가? (2) D1–D3의 관계 질문에 top-k=2로 어떤 청크가 회수되는가?


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from session_demo.data import Document, policy_chunks, project_documents, policy_coverage, relation_coverage
from session_demo.vector import ChromaTfidfIndex, hit_rows
print("자료 폴더:", ROOT / "data")


## 1. 정책 예외가 청크 사이에 나뉜 경우

먼저 원문 A·B·C를 모두 읽어 봅니다. 질문은 **“2025년 4월 가입한 VIP 고객의 해외 송금 수수료는 면제되는가?”**입니다. 원문 전체의 정답은 **면제되지 않는다**입니다. B가 해외 송금의 예외를 명시하기 때문입니다.


In [ ]:
policy = policy_chunks()
display(pd.DataFrame([{"ID": d.id, "문장": d.text} for d in policy]))
policy_question = "2025년 4월 가입한 VIP 고객의 해외 송금 수수료는 면제돼?"
index_policy = ChromaTfidfIndex(policy)
policy_hits = index_policy.search(policy_question, k=2)
display(pd.DataFrame(hit_rows(policy_hits)))
print("실제 회수 ID:", [h.id for h in policy_hits])
policy_found = policy_coverage([h.id for h in policy_hits])
print("누락된 청크:", policy_found["missing"])


**검색 결과를 그대로 읽으세요.** 위의 `누락된 청크`는 답변 결과가 아니라 검색되지 않은 근거입니다. 검색기의 순위가 어떻게 나왔든 top-k=2에는 세 청크가 모두 들어갈 수 없습니다. 다만 `B`가 빠졌다는 결과와 `A`나 `C`가 빠졌다는 결과는 답변에 미치는 영향이 다릅니다. 아래 A·C만 제공하는 장면은 **의도적으로 만든 반례**이며, 위 Chroma의 실제 순위라고 주장하지 않습니다.


In [ ]:
controlled_ids = ["A", "C"]
controlled = [d for d in policy if d.id in controlled_ids]
display(pd.DataFrame([{"제공된 ID": d.id, "LLM에 보이는 문장": d.text} for d in controlled]))
controlled_coverage = policy_coverage(controlled_ids)
print("제공된 청크:", controlled_coverage["retrieved"])
print("누락된 예외 청크:", controlled_coverage["missing"])
print("A·C만 보고 일반 규칙을 적용한 답(가능한 오답, LLM 출력 아님): 면제된다")
print("A·B·C 전체 정책의 정답: 면제되지 않는다 [B]")


A·C만 보고 ‘면제된다’고 답하면 전체 정책의 예외 B와 충돌합니다. 위의 오답은 **누락된 근거의 위험을 보여주기 위해 구성한 예시**이며 실제 LLM 응답은 아닙니다. 모델이 반드시 이렇게 답한다는 뜻도 아닙니다.

### 같은 RAG에서도 보완할 수 있다

실제 검색에서 관련 청크 하나를 찾으면 **부모 정책 문서 전체**나 **앞뒤 이웃 청크**를 함께 회수하도록 설계할 수 있습니다. 아래는 세 문장을 부모 문서 한 단위로 색인한 단순한 대안입니다. 이 사례만으로 그래프가 필요하다고 결론 내리지 않습니다.


In [ ]:
parent = Document("POLICY", "VIP 정책 전체", " ".join(f"[{d.id}] {d.text}" for d in policy))
parent_hits = ChromaTfidfIndex([parent]).search(policy_question, k=1)
display(pd.DataFrame(hit_rows(parent_hits)))
print("부모 문서 안에는 A·B·C가 모두 들어 있습니다.")


## 2. 세 문서의 관계를 물으면 top-k=2에서 무엇이 빠질까?

자료는 D1 담당자, D2 프로그램 소속, D3 보안팀 역할입니다. 세 사실을 모두 연결해야 **오헤슬과 보안팀이 같은 프로그램을 통해 어떤 관계인지** 설명할 수 있습니다. 직접 협업을 추론할 근거는 없습니다.


In [ ]:
docs = project_documents(("D1", "D2", "D3"))
display(pd.DataFrame([{"ID": d.id, "제목": d.title, "원문": d.text} for d in docs]))
relation_question = "오헤슬과 보안팀은 어떤 관계인가?"
index_relation = ChromaTfidfIndex(docs)
top2 = index_relation.search(relation_question, k=2)
display(pd.DataFrame(hit_rows(top2)))
print("실제 top-2 ID:", [h.id for h in top2])
print("근거 범위:", relation_coverage([h.id for h in top2]))


`missing`에 나온 문서가 왜 필요한지 해당 원문을 다시 확인해 봅니다. **top-k=2가 늘 특정 두 문서를 고른다고 가정하지 않습니다.** 검색 방식·질문 표현·임베딩 모델이 바뀌면 순위도 달라집니다.

`k=3`이면 이 작은 세 문서 예제에서는 모든 근거가 들어옵니다. 실제 시스템에서는 이웃·부모 회수, 질문 분해, 하이브리드 검색, 재순위화도 후보입니다.


In [ ]:
top3 = index_relation.search(relation_question, k=3)
display(pd.DataFrame(hit_rows(top3)))
print("top-3 근거 범위:", relation_coverage([h.id for h in top3]))


## 3. 생성 단계는 선택적으로 실행

아래를 `True`로 바꾸면 **실제 회수된 문장만** LLM에 보냅니다. API 비용이 발생합니다. `top2`와 `top3`의 답을 비교할 때 모델의 표현 차이보다 먼저 제공된 근거 ID를 확인합니다.


In [ ]:
RUN_LLM = False
if RUN_LLM:
    from session_demo.llm import grounded_answer
    by_id = {d.id: d for d in docs}
    for label, hits in [("top-2", top2), ("top-3", top3)]:
        evidence = [by_id[h.id] for h in hits]
        print(f"\n[{label}] 제공 ID: {[d.id for d in evidence]}")
        print(grounded_answer(relation_question, evidence))
else:
    print("LLM 호출을 건너뛰었습니다. 검색 근거 비교는 이미 실행되었습니다.")


**생성 답변도 근거와 대조:** D3는 보안팀이 *프로그램*을 검토한다고만 말합니다. 모델이 이를 "소속 프로젝트를 직접 검토한다"로 확대했다면 그 문장은 근거를 넘은 추론입니다. top-3에 세 문서가 모두 들어와도 답변 자체를 검토해야 합니다.

**다음:** `02_graph_neo4j.ipynb`에서는 D1–D3를 명시적 관계로 보관하고, 질문에 맞는 경로와 각 선의 원문을 회수합니다. 그 그래프는 수업용으로 사람이 확인해 입력한 것입니다.
